# 🎥 Vynora: Creator-Aware Micro-Video Recommendation with Multi-Objective Diversification
### Bachelor's Thesis Project (BTP) — 5th Semester
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yshasvee11/Video-Recommendation-GNN/blob/main/vynora2_colab.ipynb)

A research implementation for micro-video recommendation incorporating:
- **VA-GNN Core**: Exact Bilinear Adaptive Gate ($w = \sigma(e_v^T Q e_{p_v})$) & 3-Entity Cross-View CL ($U, V, P$)
- **Sagittarius Adaptation**: Log-frequency interaction weighting ($w_{uv} = \log(1 + c_{uv})$)
- **MPGCL Extension**: Sparse homogeneous meta-path regularizers ($A_U = RR^T, A_V = R^TR$)
- **CONDE Adaptation**: Differentiable soft relevance attention
- **Paper 5 Diversification**: Calibrated publisher & embedding-space diversity with MMR and Fair Re-ranking

---
### ⚡ Hardware Acceleration Check
1. Go to **Runtime** > **Change runtime type**
2. Under **Hardware accelerator**, select **T4 GPU**
3. Click **Save**

In [ ]:
# 1. Verify T4 GPU is active
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name   :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM    : {vram:.2f} GB')
else:
    print('WARNING: CUDA is not active. Switch to T4 GPU in Runtime > Change runtime type.')

## 📦 Step 1: Setup Repository and Bundled Datasets
Clones the repository and verifies that both the `takatak` and `wechat` benchmark datasets are ready.

In [ ]:
import os
REPO_DIR = '/content/Video-Recommendation-GNN'

# 1. Clone repository if running in a fresh Colab session
if not os.path.exists(REPO_DIR):
    print('Cloning Video-Recommendation-GNN repository...')
    !git clone https://github.com/yshasvee11/Video-Recommendation-GNN.git {REPO_DIR}
else:
    print('Repository already present. Pulling latest updates...')
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# 2. Verify bundled datasets
for ds in ['takatak', 'wechat']:
    p = os.path.join('data', ds, 'user_video_data.txt')
    if os.path.exists(p):
        print(f'✓ Dataset "{ds}" ready at data/{ds}')
    else:
        print(f'✗ Warning: "{ds}" missing at {p}')

# 3. Install requirements
!pip install -q torch numpy scipy tqdm matplotlib pandas
print('✓ Environment setup complete!')

## 🚀 Step 2: Train the Recommender Model
Train the model with the **VA-GNN Bilinear Adaptive Gate** ($w = \sigma(e_v^T Q e_{p_v})$), **3-Entity Contrastive Learning** ($U, V, P$), and **Sagittarius Frequency-Weighted BPR**.

In [ ]:
#@title ⚙️ Training Configuration { run: 'auto' }
DATASET = 'takatak' #@param ['takatak', 'wechat']
EPOCHS = 15 #@param {type:'slider', min:1, max:50, step:1}
LR = 0.001 #@param {type:'number'}
USE_ADAPTIVE_GATE = True #@param {type:'boolean'}
USE_3ENTITY_CL = True #@param {type:'boolean'}
USE_FREQUENCY_WEIGHTING = True #@param {type:'boolean'}
CL_REG = 0.1 #@param {type:'number'}
VLOGGER_REG = 0.1 #@param {type:'number'}

# Launch training
!python train.py --dataset {DATASET}

## 📈 Step 3: Visualize Training Dynamics
Plot the BPR loss progression and validation Recall@10 across training epochs.

In [ ]:
#@title 📊 Plot Loss & Recall Curves
import json
import os
import matplotlib.pyplot as plt

result_file = f'results/results_{DATASET}.json'
if os.path.exists(result_file):
    with open(result_file, 'r') as f:
        res_data = json.load(f)

    history = res_data.get('history', [])
    epochs = [h['epoch'] for h in history]
    losses = [h['loss'] for h in history]
    recalls = [h.get('val_Recall@10', 0.0) for h in history]
    ndcgs = [h.get('val_NDCG@10', 0.0) for h in history]
    best_ep = res_data.get('best_epoch', 1)
    best_rec = res_data.get('val_recall', 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    # Loss curve
    axes[0].plot(epochs, losses, 'o-', color='#e74c3c', linewidth=2, label='Training Loss')
    axes[0].set_xlabel('Epoch', fontsize=11)
    axes[0].set_ylabel('Loss', fontsize=11)
    axes[0].set_title(f'BPR Loss Curve ({DATASET.upper()})', fontsize=13, fontweight='bold')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(fontsize=10)

    # Validation metrics
    axes[1].plot(epochs, recalls, 's-', color='#2980b9', linewidth=2, label='Val Recall@10')
    axes[1].plot(epochs, ndcgs, '^-', color='#27ae60', linewidth=2, label='Val NDCG@10')
    if best_ep in epochs:
        axes[1].scatter([best_ep], [best_rec], color='#f39c12', s=160, zorder=5, marker='*',
                        label=f'Best Checkpoint (Ep {best_ep}: {best_rec:.4f})')
    axes[1].set_xlabel('Epoch', fontsize=11)
    axes[1].set_ylabel('Score', fontsize=11)
    axes[1].set_title(f'Validation Progression ({DATASET.upper()})', fontsize=13, fontweight='bold')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f'\n✓ Best Checkpoint: Epoch {best_ep} | Val Recall@10 = {best_rec:.4f}')
    t_rec = res_data.get('test_results', {}).get('Recall', {}).get('10', 'N/A')
    print(f'  Final Test Recall@10: {t_rec}')
else:
    print(f'Results not found at {result_file}. Run Step 2 training first.')

## 🔬 Step 4: Run Accuracy vs. Diversity Benchmark
Sweep the diversity trade-off $\lambda \in [0.0, 1.0]$ using **Maximal Marginal Relevance (MMR)** to evaluate the Pareto trade-off between recommendation accuracy and creator diversity.

In [ ]:
#@title 🔬 Run Accuracy vs. Diversity Sweep { run: 'auto' }
BENCHMARK_MODE = 'val' #@param ['val', 'test']
DIVERSITY_TYPE = 'publisher' #@param ['publisher', 'content', 'hybrid']
DIVERSITY_METHOD = 'mmr' #@param ['mmr', 'score_rerank', 'creator_cap']

!python diversity_benchmark.py --dataset {DATASET} --mode {BENCHMARK_MODE} --type {DIVERSITY_TYPE} --method {DIVERSITY_METHOD}

## ⚖️ Step 5: Fair Re-Ranking Comparison Study
Compares 6 methods from the **identical Top-100 candidate pool** produced by the trained VA-GNN backbone:
1. **Pure Relevance Baseline** ($\lambda = 0.0$)
2. **Hard Creator Cap** (Max 2 videos per creator in Top-10)
3. **MMR Publisher Diversity** ($\lambda = 0.3$)
4. **MMR Embedding-Space Diversity** ($\lambda = 0.3$)
5. **Hybrid MMR** (0.5 Publisher + 0.5 Embedding)
6. **Empirical Score Blending** ($\lambda = 0.3$)

In [ ]:
#@title ⚖️ Run Fair Re-Ranking Comparison Study
!python diversity_benchmark.py --dataset {DATASET} --mode val --compare

## 📊 Step 6: Visualize Accuracy vs. Diversity Pareto Frontier
Display the Pareto frontier curve showing how publisher diversity can be increased significantly while preserving or even improving Recall@10.

In [ ]:
#@title 📊 Pareto Frontier and Coverage Plots
import json
import os
import matplotlib.pyplot as plt
import pandas as pd

tradeoff_file = f'results/diversity_tradeoff_{DATASET}_{DIVERSITY_METHOD}.json'
if os.path.exists(tradeoff_file):
    with open(tradeoff_file, 'r') as f:
        bench_data = json.load(f)

    df = pd.DataFrame(bench_data)

    # Summary table
    print('='*78)
    print(f'  ACCURACY VS. DIVERSITY BENCHMARK TABLE ({DATASET.upper()})')
    print('='*78)
    cols = ['lambda', 'Recall@10', 'NDCG@10', 'Publisher_Diversity@10', 'Catalog_Coverage@10']
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print('='*78)

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    lambdas = df['lambda'].values
    recalls = df['Recall@10'].values
    pub_divs = df['Publisher_Diversity@10'].values
    coverages = df['Catalog_Coverage@10'].values

    # Subplot 1: Pareto curve (Recall vs PubDiv)
    axes[0].plot(pub_divs, recalls, 'o-', color='#8e44ad', linewidth=2.5, markersize=8)
    for lam, x, y in zip(lambdas, pub_divs, recalls):
        axes[0].annotate(f'λ={lam:.1f}', (x, y), textcoords='offset points', xytext=(6, 5),
                         fontsize=9, fontweight='bold', color='#2c3e50')
    axes[0].set_xlabel('Publisher Diversity @ 10 (Gini-Simpson)', fontsize=11)
    axes[0].set_ylabel('Recall @ 10', fontsize=11)
    axes[0].set_title('Accuracy vs. Publisher Diversity Pareto Curve', fontsize=13, fontweight='bold')
    axes[0].grid(True, linestyle='--', alpha=0.6)

    # Highlight baseline vs sweet spot
    axes[0].scatter([pub_divs[0]], [recalls[0]], color='#e74c3c', s=130, zorder=5,
                    label=f'Baseline (λ=0.0): R@10={recalls[0]:.4f}')
    if len(pub_divs) > 3:
        axes[0].scatter([pub_divs[3]], [recalls[3]], color='#27ae60', s=140, zorder=5,
                        label=f'MMR (λ=0.3): R@10={recalls[3]:.4f}, Div={pub_divs[3]:.4f}')
    axes[0].legend(fontsize=10)

    # Subplot 2: Progression vs lambda
    ax2 = axes[1]
    ax2_r = ax2.twinx()
    l1 = ax2.plot(lambdas, pub_divs, 's-', color='#2980b9', linewidth=2, label='Publisher Diversity @ 10')
    l2 = ax2_r.plot(lambdas, coverages, '^-', color='#d35400', linewidth=2, label='Catalog Coverage @ 10')
    ax2.set_xlabel('Diversity Trade-off Weight (λ)', fontsize=11)
    ax2.set_ylabel('Publisher Diversity @ 10', color='#2980b9', fontsize=11)
    ax2_r.set_ylabel('Catalog Coverage @ 10', color='#d35400', fontsize=11)
    ax2.set_title('Diversity & Catalog Coverage across Trade-off λ', fontsize=13, fontweight='bold')
    ax2.grid(True, linestyle='--', alpha=0.6)

    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center left', fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print(f'Benchmark file not found at {tradeoff_file}. Run Step 4 first.')

## 📱 Step 7: Interactive Single-User Home Screen Recommendation
Generate personalized top-10 recommendations for any user ID, comparing the **Baseline (\lambda = 0.0)** against the **Diversified Feed (\lambda = 0.4)**. Displays a creator distribution bar chart showing how MMR eliminates creator clustering.

In [ ]:
#@title 📱 User Feed Recommendation Demo { run: 'auto' }
DEMO_USER_ID = 42 #@param {type:'integer'}
DIVERSITY_WEIGHT = 0.4 #@param {type:'slider', min:0.0, max:1.0, step:0.05}
TOP_K = 10 #@param {type:'integer'}

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from predict import load_model_and_dataset, recommend_for_user

model, dataset, graph_uv, graph_ua = load_model_and_dataset(DATASET)

# 1. Baseline recommendations (lambda = 0.0)
base_out = recommend_for_user(
    user_id=DEMO_USER_ID,
    model=model,
    dataset=dataset,
    graph_uv=graph_uv,
    graph_ua=graph_ua,
    topk=TOP_K,
    diversity_lambda=0.0
)

# 2. Diversified recommendations
div_out = recommend_for_user(
    user_id=DEMO_USER_ID,
    model=model,
    dataset=dataset,
    graph_uv=graph_uv,
    graph_ua=graph_ua,
    topk=TOP_K,
    diversity_lambda=DIVERSITY_WEIGHT,
    diversity_type='publisher',
    diversity_method='mmr'
)

base_recs = base_out['results']
div_recs = div_out['results']

print(f'\n{"="*72}')
print(f'  FEED RECOMMENDATION COMPARISON FOR USER {DEMO_USER_ID} ({DATASET.upper()})')
print(f'{"="*72}')
print(f'{"Rank":<5} | {"Baseline (λ=0.0) Video [Vlogger]":<30} | {"Diversified (λ="+str(DIVERSITY_WEIGHT)+") Video [Vlogger]":<32}')
print(f'{"-"*72}')
for r in range(min(TOP_K, len(base_recs))):
    bv = base_recs[r]
    dv = div_recs[r]
    b_str = f"Video {bv['video_id']:>5d} (Creator {bv['vlogger_id']:>4d})"
    d_str = f"Video {dv['video_id']:>5d} (Creator {dv['vlogger_id']:>4d})"
    print(f"{r+1:<5} | {b_str:<30} | {d_str:<32}")
print(f'{"="*72}\n')

bm = base_out['diversity_metrics']
dm = div_out['diversity_metrics']
diff = dm['publisher_diversity'] - bm['publisher_diversity']
print(f"Publisher Diversity (Eq 1):  {bm['publisher_diversity']:.4f}  →  {dm['publisher_diversity']:.4f} ({diff*100:+.1f}%)")
print(f"Unique Creators in Top-{TOP_K}:    {bm['unique_vloggers']} / {TOP_K}       →  {dm['unique_vloggers']} / {TOP_K}")
print(f"Creator Shannon Entropy:     {bm['vlogger_entropy']:.3f} bits →  {dm['vlogger_entropy']:.3f} bits\n")

# Creator frequency bar chart
bv_ids = [x['vlogger_id'] for x in base_recs]
dv_ids = [x['vlogger_id'] for x in div_recs]
b_c = Counter(bv_ids)
d_c = Counter(dv_ids)
all_creators = sorted(list(set(bv_ids + dv_ids)))
xp = np.arange(len(all_creators))
w = 0.38

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(xp - w/2, [b_c.get(c, 0) for c in all_creators], width=w, label=f'Baseline (λ=0.0): {len(b_c)} creators', color='#e74c3c')
ax.bar(xp + w/2, [d_c.get(c, 0) for c in all_creators], width=w, label=f'Diversified (λ={DIVERSITY_WEIGHT}): {len(d_c)} creators', color='#2ecc71')
ax.set_xticks(xp)
ax.set_xticklabels([f'Creator {c}' for c in all_creators], rotation=45, ha='right')
ax.set_ylabel('Recommendation Count in Top-10', fontsize=11)
ax.set_title(f'Creator Frequency in Top-{TOP_K} for User {DEMO_USER_ID}', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5, axis='y')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 🧪 Step 8: Run 8-Variant Ablation Suite (M1 through M8)
Evaluates the 8 publication ablation variants:
- **M1**: LightGCN Baseline
- **M2**: VA-GNN (Fixed Gate $\lambda=0.5$)
- **M3**: VA-GNN (Bilinear Adaptive Gate $Q$)
- **M4**: VA-GNN Core (+ 3-Entity CL: User + Video + Creator)
- **M5**: VA-GNN + Sagittarius Frequency-Weighted BPR
- **M6**: VA-GNN + MPGCL Meta-Paths (UVU & VUV)
- **M7**: VA-GNN + Soft Interaction Filtering
- **M8**: Full Vynora + MMR Diversity

In [ ]:
#@title 🧪 Run 8-Variant Ablation Suite
RUN_ABLATION = False #@param {type:'boolean'}

if RUN_ABLATION:
    !python ablation.py
    import json
    if os.path.exists('results/ablation_summary.json'):
        with open('results/ablation_summary.json', 'r') as f:
            abl = json.load(f)
        import pandas as pd
        rows = []
        for var, vals in abl.items():
            r10 = vals.get('Recall', {}).get('10', 0.0)
            n10 = vals.get('NDCG', {}).get('10', 0.0)
            rows.append({'Variant': var, 'Recall@10': r10, 'NDCG@10': n10})
        print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Ablation skipped. Set RUN_ABLATION = True and re-run cell to execute.')

## 💾 Step 9: Export and Download Experiment Checkpoints & Results
Package trained model checkpoints, training logs, and benchmark JSON files into a zip archive and download directly to your computer.

In [ ]:
#@title 💾 Download Checkpoints and Results Zip
from google.colab import files
import os

zip_name = f'vynora_experiment_results_{DATASET}.zip'
!zip -r {zip_name} checkpoints/ results/
if os.path.exists(zip_name):
    print(f'✓ Created {zip_name}. Triggering browser download...')
    files.download(zip_name)